# Slow-recovery probability from structure and pumping

Mainline workflow only: the Slow recovery class is defined from drought-response clustering, then the probability of Slow recovery is predicted from the native AEM resistivity profile and pumping intensity with an ExtraTrees classifier. No drought-response metrics are used as predictors.

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from pandas.errors import PerformanceWarning
from rasterio.transform import xy as raster_xy
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

try:
    import geopandas as gpd
    from pyproj import Transformer
except ImportError:
    gpd = None
    Transformer = None

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=PerformanceWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023.')


def display_path(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
CLUSTER_LABELS_CSV = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
AEM_LOG10RES_TIF = ROOT / 'data' / '1 resistivity' / 'aem_log10res_1km_masked.tif'
AEM_DEPTH_LEVELS_JSON = ROOT / 'data' / '1 resistivity' / 'aem_depth_levels_m.json'
PUMPING_MONTHLY_CSV = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H1' / 'aiwum_monthly.csv'
MRVA_BOUNDARY_PATH = ROOT / 'assets' / 'spatial' / 'mrva_boundary.geojson'
MISSISSIPPI_RIVER_GMT_PATH = ROOT / 'assets' / 'spatial' / 'mississippi_river.gmt'

OUT_DIR = RECON / 'metrics' / 'regression'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PROBABILITY_CSV = OUT_DIR / 'best_extratrees_slow_recovery_probabilities.csv'
IMPORTANCE_CSV = OUT_DIR / 'best_extratrees_slow_recovery_feature_importance.csv'
SUMMARY_JSON = OUT_DIR / 'best_extratrees_slow_recovery_summary.json'

HIGH_LOGRHO_THRESHOLD = 1.50  # about 31.6 ohm m
LOW_LOGRHO_THRESHOLD = 1.00   # about 10 ohm m
BLOCK_SIZE_KM = 50
N_SPATIAL_FOLDS = 5
RANDOM_SEED = 42
EXPORT_DPI = 900

CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_SHORT = {
    'Fast recovery': 'F',
    'Slow recovery': 'S',
    'Buffered': 'B',
}
TARGET_CLASS = 'Slow recovery'
TARGET_SHORT = 'Slow'
TARGET_PROBABILITY_COLUMN = 'P_slow_recovery_extratrees_resistivity_pumping'
TARGET_OOF_COLUMN = 'P_slow_recovery_extratrees_oof'
TARGET_FLAG_COLUMN = 'is_slow_recovery_class'
TARGET_HIGH_COLUMN = 'high_P_slow_recovery_equal_area_oof'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'axes.linewidth': 0.75,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
})

print('Cluster labels:', display_path(CLUSTER_LABELS_CSV))
print('AEM raster:', display_path(AEM_LOG10RES_TIF))
print('Pumping table:', display_path(PUMPING_MONTHLY_CSV))
print('Output:', display_path(OUT_DIR))

## Load response labels and AEM profiles

In [ ]:
labels = pd.read_csv(CLUSTER_LABELS_CSV)
required = {'grid_id', 'row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class'}
missing = required.difference(labels.columns)
if missing:
    raise KeyError(f'Missing required label columns: {sorted(missing)}')

labels['valid_for_clustering'] = labels['valid_for_clustering'].astype(bool)
rows = labels['row'].to_numpy(dtype=np.int64)
cols = labels['col'].to_numpy(dtype=np.int64)
valid_label_mask = labels['valid_for_clustering'].to_numpy() & labels['response_class'].notna().to_numpy()

with AEM_DEPTH_LEVELS_JSON.open('r', encoding='utf-8') as f:
    native_depths_m = np.asarray(json.load(f), dtype=np.float32)

with rasterio.open(AEM_LOG10RES_TIF) as src:
    if src.count != len(native_depths_m):
        raise ValueError(f'AEM raster has {src.count} bands, but depth file has {len(native_depths_m)} levels.')

    # The label table is south-to-north, while the GeoTIFF rows are north-to-south.
    raster_rows = (src.height - 1) - rows
    inside = (raster_rows >= 0) & (raster_rows < src.height) & (cols >= 0) & (cols < src.width)
    if not np.all(inside):
        raise ValueError('Some label row/col indices fall outside the AEM raster.')

    check_idx = np.linspace(0, len(labels) - 1, min(1000, len(labels)), dtype=np.int64)
    check_x, check_y = raster_xy(src.transform, raster_rows[check_idx], cols[check_idx], offset='center')
    checked_xy = np.column_stack([check_x, check_y])
    target_xy = labels.loc[check_idx, ['x', 'y']].to_numpy(dtype=float)
    coord_error_m = np.nanmax(np.abs(checked_xy - target_xy))
    if coord_error_m > 1e-3:
        raise ValueError(f'AEM/sample coordinate mismatch: max sampled error = {coord_error_m:.3f} m')

    aem_stack = src.read().astype(np.float32)
    if src.nodata is not None:
        aem_stack[aem_stack == src.nodata] = np.nan
    log10_profiles_native = aem_stack[:, raster_rows, cols].T.astype(np.float32)

print(f'Coordinate check max error: {coord_error_m:.6f} m')
print(f'AEM profile matrix: {log10_profiles_native.shape[0]:,} cells x {log10_profiles_native.shape[1]} layers')
print(f'Native AEM depth range: {native_depths_m.min():.1f}-{native_depths_m.max():.1f} m')
print(f'Valid response-labelled cells: {valid_label_mask.sum():,}')

## Build structure and pumping predictors

In [ ]:
depth_grid_m = np.arange(0.0, min(400.0, float(np.floor(native_depths_m.max()))) + 1.0, 1.0, dtype=np.float32)
log10_profiles_1m = np.full((len(labels), len(depth_grid_m)), np.nan, dtype=np.float32)

for i, profile in enumerate(log10_profiles_native):
    good = np.isfinite(profile) & np.isfinite(native_depths_m)
    if good.sum() >= 2:
        log10_profiles_1m[i] = np.interp(
            depth_grid_m,
            native_depths_m[good],
            profile[good],
            left=np.nan,
            right=np.nan,
        ).astype(np.float32)


def depth_mask(zmin: float, zmax: float) -> np.ndarray:
    return (depth_grid_m >= zmin) & (depth_grid_m < zmax)


def band_mean(zmin: float, zmax: float) -> np.ndarray:
    mask = depth_mask(zmin, zmax)
    if not np.any(mask):
        return np.full(len(labels), np.nan, dtype=np.float32)
    return np.nanmean(log10_profiles_1m[:, mask], axis=1).astype(np.float32)


def thickness_m(mask_2d: np.ndarray) -> np.ndarray:
    return np.sum(mask_2d, axis=1).astype(np.float32)


def max_continuous_thickness_m(mask_2d: np.ndarray) -> np.ndarray:
    out = np.zeros(mask_2d.shape[0], dtype=np.float32)
    for i, arr in enumerate(mask_2d):
        if not arr.any():
            continue
        transitions = np.diff(np.r_[False, arr, False].astype(np.int8))
        starts = np.where(transitions == 1)[0]
        ends = np.where(transitions == -1)[0]
        if starts.size:
            out[i] = float((ends - starts).max())
    return out


features = labels[['grid_id', 'row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class']].copy()

FINE_DEPTH_BANDS_M = [
    (0, 15), (15, 30), (30, 50), (50, 100),
    (100, 150), (150, 200), (200, 300), (300, 400),
]
FINE_RESISTIVITY_FEATURES = []
for zmin, zmax in FINE_DEPTH_BANDS_M:
    col = f'logrho_{zmin}_{zmax}_mean'
    features[col] = band_mean(zmin, zmax)
    FINE_RESISTIVITY_FEATURES.append(col)

features['logrho_shallow_0_50_mean'] = band_mean(0, 50)
features['logrho_middle_50_150_mean'] = band_mean(50, 150)
features['logrho_deep_150_300_mean'] = band_mean(150, 300)

high_mask = (log10_profiles_1m >= HIGH_LOGRHO_THRESHOLD) & depth_mask(0, 400)
low_mask = (log10_profiles_1m <= LOW_LOGRHO_THRESHOLD) & depth_mask(0, 400)
features['high_res_thickness_0_400_m'] = thickness_m(high_mask)
features['low_res_thickness_0_400_m'] = thickness_m(low_mask)
features['continuous_high_res_thickness_m'] = max_continuous_thickness_m(high_mask)
features['mid_low_barrier_strength'] = np.nansum(
    np.maximum(LOW_LOGRHO_THRESHOLD - log10_profiles_1m[:, depth_mask(50, 200)], 0),
    axis=1,
).astype(np.float32)
features['sand_clay_sand_contrast'] = (
    0.5 * (features['logrho_shallow_0_50_mean'] + features['logrho_deep_150_300_mean'])
    - features['logrho_middle_50_150_mean']
).astype(np.float32)

STRUCTURE_FEATURES = [
    'logrho_shallow_0_50_mean',
    'logrho_middle_50_150_mean',
    'logrho_deep_150_300_mean',
    'high_res_thickness_0_400_m',
    'low_res_thickness_0_400_m',
    'continuous_high_res_thickness_m',
    'mid_low_barrier_strength',
    'sand_clay_sand_contrast',
]

PROFILE_SHAPE_FEATURES = []
for left, right in zip(FINE_RESISTIVITY_FEATURES[:-1], FINE_RESISTIVITY_FEATURES[1:]):
    col = f'delta_{left}_to_{right}'
    features[col] = (features[right] - features[left]).astype(np.float32)
    PROFILE_SHAPE_FEATURES.append(col)

fine_profile_matrix = features[FINE_RESISTIVITY_FEATURES].to_numpy(dtype=np.float32)
features['logrho_band_mean'] = np.nanmean(fine_profile_matrix, axis=1).astype(np.float32)
features['logrho_band_std'] = np.nanstd(fine_profile_matrix, axis=1).astype(np.float32)
features['logrho_band_range'] = (np.nanmax(fine_profile_matrix, axis=1) - np.nanmin(fine_profile_matrix, axis=1)).astype(np.float32)
features['logrho_shallow_minus_middle'] = (features['logrho_shallow_0_50_mean'] - features['logrho_middle_50_150_mean']).astype(np.float32)
features['logrho_middle_minus_deep'] = (features['logrho_middle_50_150_mean'] - features['logrho_deep_150_300_mean']).astype(np.float32)
features['logrho_shallow_minus_deep'] = (features['logrho_shallow_0_50_mean'] - features['logrho_deep_150_300_mean']).astype(np.float32)
PROFILE_SHAPE_FEATURES += [
    'logrho_band_mean',
    'logrho_band_std',
    'logrho_band_range',
    'logrho_shallow_minus_middle',
    'logrho_middle_minus_deep',
    'logrho_shallow_minus_deep',
]

NATIVE_PROFILE_FEATURES = []
for layer_idx, depth_m in enumerate(native_depths_m):
    col = f"logrho_native_{depth_m:.1f}m".replace('.', 'p')
    features[col] = log10_profiles_native[:, layer_idx].astype(np.float32)
    NATIVE_PROFILE_FEATURES.append(col)
features['native_logrho_mean'] = np.nanmean(log10_profiles_native, axis=1).astype(np.float32)
features['native_logrho_std'] = np.nanstd(log10_profiles_native, axis=1).astype(np.float32)
features['native_logrho_range'] = (np.nanmax(log10_profiles_native, axis=1) - np.nanmin(log10_profiles_native, axis=1)).astype(np.float32)
NATIVE_PROFILE_FEATURES += ['native_logrho_mean', 'native_logrho_std', 'native_logrho_range']

finite_native = np.isfinite(log10_profiles_native)
features['valid_aem_layer_fraction'] = finite_native.mean(axis=1).astype(np.float32)
features['max_valid_aem_depth_m'] = np.nan
has_any_native = finite_native.any(axis=1)
features.loc[has_any_native, 'max_valid_aem_depth_m'] = [
    float(native_depths_m[finite_native[i]].max()) for i in np.where(has_any_native)[0]
]


def load_pumping_features(path: Path) -> pd.DataFrame:
    pumping = pd.read_csv(path, usecols=['grid_id', 'month_label', 'monthly_pumping_mm'])
    pumping['month_label'] = pumping['month_label'].astype(str)
    pumping['year'] = pumping['month_label'].str[:4].astype(int)
    pumping['month'] = pumping['month_label'].str[5:7].astype(int)

    base = pumping.groupby('grid_id', as_index=False).agg(
        pumping_mean_monthly_2011_2023_mm=('monthly_pumping_mm', 'mean'),
        pumping_peak_month_2011_2023_mm=('monthly_pumping_mm', 'max'),
    )
    annual = pumping.groupby(['grid_id', 'year'], as_index=False)['monthly_pumping_mm'].sum()
    annual_mean = annual.groupby('grid_id', as_index=False)['monthly_pumping_mm'].mean().rename(
        columns={'monthly_pumping_mm': 'pumping_mean_annual_2011_2023_mm_per_year'}
    )
    growing = pumping[pumping['month'].between(4, 10)].groupby(['grid_id', 'year'], as_index=False)['monthly_pumping_mm'].sum()
    growing_mean = growing.groupby('grid_id', as_index=False)['monthly_pumping_mm'].mean().rename(
        columns={'monthly_pumping_mm': 'pumping_mean_growing_2011_2023_mm_per_year'}
    )
    drought_2012 = pumping[
        (pumping['year'] == 2012) & pumping['month'].between(5, 10)
    ].groupby('grid_id', as_index=False)['monthly_pumping_mm'].sum().rename(
        columns={'monthly_pumping_mm': 'pumping_2012_drought_MayOct_mm'}
    )

    out = base.merge(annual_mean, on='grid_id', how='left')
    out = out.merge(growing_mean, on='grid_id', how='left')
    out = out.merge(drought_2012, on='grid_id', how='left')
    for col in out.columns:
        if col != 'grid_id':
            out[col] = out[col].fillna(0).astype(np.float32)
    return out


pumping_features = load_pumping_features(PUMPING_MONTHLY_CSV)
features = features.merge(pumping_features, on='grid_id', how='left')

PUMPING_FEATURES = [
    'pumping_2012_drought_MayOct_mm',
    'pumping_mean_growing_2011_2023_mm_per_year',
    'pumping_mean_annual_2011_2023_mm_per_year',
    'pumping_peak_month_2011_2023_mm',
]
LOG_PUMPING_FEATURES = []
for col in PUMPING_FEATURES:
    features[col] = features[col].fillna(0).astype(np.float32)
    log_col = f'log1p_{col}'
    features[log_col] = np.log1p(np.clip(features[col].to_numpy(dtype=float), a_min=0, a_max=None)).astype(np.float32)
    LOG_PUMPING_FEATURES.append(log_col)

FEATURE_COLUMNS = NATIVE_PROFILE_FEATURES + LOG_PUMPING_FEATURES

print('Predictor count:', len(FEATURE_COLUMNS))
print('\nAEM coverage by depth band among response-labelled cells:')
for zmin, zmax in FINE_DEPTH_BANDS_M:
    col = f'logrho_{zmin}_{zmax}_mean'
    frac = np.isfinite(features.loc[valid_label_mask, col]).mean()
    print(f'  {zmin:3d}-{zmax:3d} m: {frac:.3f}')
print('\nResponse-class counts:')
print(features.loc[valid_label_mask, 'response_class'].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int).to_string())

## Spatial-block validation and final ExtraTrees model

In [ ]:
has_aem_predictor = np.any(np.isfinite(features[FINE_RESISTIVITY_FEATURES].to_numpy(dtype=float)), axis=1)
model_mask = valid_label_mask & has_aem_predictor
X = features.loc[model_mask, FEATURE_COLUMNS].to_numpy(dtype=float)
y = (features.loc[model_mask, 'response_class'] == TARGET_CLASS).to_numpy(dtype=int)
coords_xy = features.loc[model_mask, ['x', 'y']].to_numpy(dtype=float)

block_size_m = BLOCK_SIZE_KM * 1000.0
block_x = np.floor((coords_xy[:, 0] - coords_xy[:, 0].min()) / block_size_m).astype(int)
block_y = np.floor((coords_xy[:, 1] - coords_xy[:, 1].min()) / block_size_m).astype(int)
groups = block_x * 1000 + block_y

selected_model = make_pipeline(
    SimpleImputer(strategy='median'),
    ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=10,
        max_features=0.75,
        class_weight=None,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
)

oof_pred = np.full(len(y), np.nan, dtype=np.float32)
fold_rows = []
splitter = GroupKFold(n_splits=N_SPATIAL_FOLDS)
for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
    estimator = clone(selected_model)
    estimator.fit(X[train_idx], y[train_idx])
    pred = estimator.predict_proba(X[test_idx])[:, 1]
    oof_pred[test_idx] = pred.astype(np.float32)
    fold_rows.append({
        'fold': fold,
        'n_test': int(len(test_idx)),
        'target_prevalence_test': float(y[test_idx].mean()),
        'roc_auc': float(roc_auc_score(y[test_idx], pred)) if len(np.unique(y[test_idx])) > 1 else np.nan,
        'average_precision': float(average_precision_score(y[test_idx], pred)),
        'brier_score': float(brier_score_loss(y[test_idx], pred)),
    })

if not np.all(np.isfinite(oof_pred)):
    raise RuntimeError('OOF prediction contains missing values; check spatial folds.')

equal_area_cutoff = float(np.nanquantile(oof_pred, 1 - y.mean()))
high_oof = oof_pred >= equal_area_cutoff
true_target = y.astype(bool)
tp = int(np.sum(high_oof & true_target))
fp = int(np.sum(high_oof & ~true_target))
fn = int(np.sum(~high_oof & true_target))

summary = {
    'method': 'ExtraTreesClassifier',
    'target': TARGET_CLASS,
    'feature_set': 'native_aem_profile_plus_pumping',
    'interpretation': 'P(Slow recovery) is estimated from the native AEM resistivity profile and pumping intensity only; drought-response metrics, coordinates and hand-built pumping-structure interactions are not used as predictors.',
    'spatial_block_km': BLOCK_SIZE_KM,
    'n_spatial_folds': N_SPATIAL_FOLDS,
    'n_model_cells': int(len(y)),
    'target_prevalence': float(y.mean()),
    'roc_auc': float(roc_auc_score(y, oof_pred)),
    'average_precision': float(average_precision_score(y, oof_pred)),
    'brier_score': float(brier_score_loss(y, oof_pred)),
    'equal_area_cutoff': equal_area_cutoff,
    'equal_area_precision': float(tp / max(tp + fp, 1)),
    'equal_area_recall': float(tp / max(tp + fn, 1)),
    'tp': tp,
    'fp': fp,
    'fn': fn,
    'features': FEATURE_COLUMNS,
    'feature_groups': {
        'native_aem_profile': NATIVE_PROFILE_FEATURES,
        'pumping': LOG_PUMPING_FEATURES,
    },
    'folds': fold_rows,
    'model_parameters': selected_model.named_steps['extratreesclassifier'].get_params(),
}

selected_model.fit(X, y)
p_final = np.full(len(features), np.nan, dtype=np.float32)
X_all = features.loc[has_aem_predictor, FEATURE_COLUMNS].to_numpy(dtype=float)
p_final[has_aem_predictor] = selected_model.predict_proba(X_all)[:, 1].astype(np.float32)

probabilities = features[['grid_id', 'row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class']].copy()
probabilities[TARGET_FLAG_COLUMN] = probabilities['response_class'].eq(TARGET_CLASS)
probabilities[TARGET_PROBABILITY_COLUMN] = p_final
probabilities[TARGET_OOF_COLUMN] = np.nan
probabilities.loc[model_mask, TARGET_OOF_COLUMN] = oof_pred
probabilities[TARGET_HIGH_COLUMN] = False
probabilities.loc[model_mask, TARGET_HIGH_COLUMN] = high_oof
probabilities.to_csv(PROBABILITY_CSV, index=False, lineterminator='\n')

forest = selected_model.named_steps['extratreesclassifier']
importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': forest.feature_importances_,
}).sort_values('importance', ascending=False)
importance.to_csv(IMPORTANCE_CSV, index=False, lineterminator='\n')
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Selected ExtraTrees spatial-block performance:')
print(
    f"ROC-AUC={summary['roc_auc']:.3f}, AP={summary['average_precision']:.3f}, "
    f"Brier={summary['brier_score']:.3f}, equal-area overlap={summary['equal_area_precision']:.3f}"
)
print(f'{TARGET_SHORT} prevalence:', f"{summary['target_prevalence']:.3f}")
print('Saved:', display_path(PROBABILITY_CSV))
print('Saved:', display_path(IMPORTANCE_CSV))
print('Saved:', display_path(SUMMARY_JSON))
print('\nTop feature importances:')
print(importance.head(12).round(4).to_string(index=False))

## Mainline figures

In [ ]:

from shapely.geometry import LineString

def save_figure(fig: mpl.figure.Figure, stem: str) -> Path:
    out = FIG_DIR / f'{stem}.png'
    fig.savefig(out, dpi=EXPORT_DPI, bbox_inches='tight')
    plt.close(fig)
    return out

def grid_edges_lonlat(label_frame: pd.DataFrame):
    x_centers = (
        label_frame.groupby('col')['x'].first()
        .sort_index().to_numpy(dtype=float)
    )
    y_centers = (
        label_frame.groupby('row')['y'].first()
        .sort_index().to_numpy(dtype=float)
    )
    dx = float(np.nanmedian(np.diff(x_centers)))
    dy = float(np.nanmedian(np.diff(y_centers)))
    x_edges_local = np.r_[
        x_centers[0] - 0.5 * dx, x_centers + 0.5 * dx
    ]
    y_edges_local = np.r_[
        y_centers[0] - 0.5 * dy, y_centers + 0.5 * dy
    ]
    xx, yy = np.meshgrid(x_edges_local, y_edges_local)
    transformer = Transformer.from_crs(
        'EPSG:5070', 'EPSG:4326', always_xy=True
    )
    lon, lat = transformer.transform(xx, yy)
    return np.asarray(lon), np.asarray(lat)

LON_EDGES, LAT_EDGES = grid_edges_lonlat(labels)
MAP_EXTENT_LONLAT = (
    float(np.nanmin(LON_EDGES)),
    float(np.nanmax(LON_EDGES)),
    float(np.nanmin(LAT_EDGES)),
    float(np.nanmax(LAT_EDGES)),
)
MAP_MEAN_LAT = 0.5 * (MAP_EXTENT_LONLAT[2] + MAP_EXTENT_LONLAT[3])
n_rows = int(labels['row'].max()) + 1
n_cols = int(labels['col'].max()) + 1

def rasterize_column(values: np.ndarray) -> np.ndarray:
    raster = np.full((n_rows, n_cols), np.nan, dtype=np.float32)
    raster[rows, cols] = values.astype(np.float32)
    return raster

def read_gmt_segments_lonlat(path: Path):
    if not path.exists():
        return []
    segments = []
    current = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                segments.append(np.asarray(current, dtype=float))
                current = []
            continue
        lon, lat = line.split()[:2]
        current.append((float(lon), float(lat)))
    if current:
        segments.append(np.asarray(current, dtype=float))
    return segments

if gpd is not None and MRVA_BOUNDARY_PATH.exists():
    mrva_boundary_lonlat = gpd.read_file(
        MRVA_BOUNDARY_PATH
    ).to_crs('EPSG:4326')
    if hasattr(mrva_boundary_lonlat.geometry, 'union_all'):
        mrva_polygon_lonlat = mrva_boundary_lonlat.geometry.union_all()
    else:
        mrva_polygon_lonlat = mrva_boundary_lonlat.unary_union
else:
    mrva_boundary_lonlat = None
    mrva_polygon_lonlat = None

mississippi_segments = read_gmt_segments_lonlat(
    MISSISSIPPI_RIVER_GMT_PATH
)

def plot_line_geometry_lonlat(ax, geometry, **kwargs):
    if geometry is None or geometry.is_empty:
        return
    if geometry.geom_type == 'LineString':
        x, y = geometry.xy
        ax.plot(np.asarray(x), np.asarray(y), **kwargs)
    elif hasattr(geometry, 'geoms'):
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part, **kwargs)

def draw_geographic_context(ax: plt.Axes):
    if mrva_boundary_lonlat is not None:
        mrva_boundary_lonlat.boundary.plot(
            ax=ax, color='#1f1f1f', linewidth=0.45, zorder=5
        )
    for segment in mississippi_segments:
        geometry = LineString(segment)
        if mrva_polygon_lonlat is not None:
            geometry = geometry.intersection(mrva_polygon_lonlat)
        plot_line_geometry_lonlat(
            ax, geometry, color='#B7DDE8', lw=0.42, zorder=4
        )
    ax.set_xlim(MAP_EXTENT_LONLAT[0], MAP_EXTENT_LONLAT[1])
    ax.set_ylim(MAP_EXTENT_LONLAT[2], MAP_EXTENT_LONLAT[3])
    ax.set_aspect(1.0 / np.cos(np.deg2rad(MAP_MEAN_LAT)))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(
        axis='both', which='both', bottom=False, left=False,
        labelbottom=False, labelleft=False
    )
    for spine in ax.spines.values():
        spine.set_visible(False)

def plot_probability_map() -> Path:
    values = probabilities[TARGET_PROBABILITY_COLUMN].to_numpy(dtype=float)
    raster = rasterize_column(values)
    valid_raster = rasterize_column(valid_label_mask.astype(np.float32)).astype(bool)
    raster[~valid_raster] = np.nan
    prob_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        'prob_slow_recovery',
        ['#F8F2E8', '#E7A3B8', '#9F2F5E'],
        N=256
    )
    fig, ax = plt.subplots(figsize=(3.35, 5.8), dpi=EXPORT_DPI)
    im = ax.pcolormesh(
        LON_EDGES, LAT_EDGES, raster,
        cmap=prob_cmap, vmin=0, vmax=1,
        shading='flat', rasterized=True
    )
    draw_geographic_context(ax)
    cbar = fig.colorbar(
        im, ax=ax, fraction=0.045, pad=0.025, extend='both'
    )
    cbar.set_label('P(Slow recovery)', rotation=90, labelpad=7)
    cbar.ax.tick_params(length=3.0, width=0.7, labelsize=9.5)
    cbar.outline.set_linewidth(0.65)
    return save_figure(
        fig, 'slow_recovery_probability_map_best_extratrees'
    )

def plot_oof_overlap_map() -> Path:
    comparison_values = np.full(
        len(probabilities), np.nan, dtype=np.float32
    )
    true_target = (
        probabilities[TARGET_FLAG_COLUMN].to_numpy(dtype=bool)
        & valid_label_mask
    )
    high_P = (
        probabilities[TARGET_HIGH_COLUMN].to_numpy(dtype=bool)
        & valid_label_mask
    )
    comparison_values[valid_label_mask] = 0
    comparison_values[true_target & ~high_P] = 1
    comparison_values[~true_target & high_P] = 2
    comparison_values[true_target & high_P] = 3
    comparison_cmap = mpl.colors.ListedColormap(
        ['#EFEFEF', '#4E6E8E', '#E1A45A', '#9F2F3C']
    )
    comparison_cmap.set_bad('#FFFFFF')
    comparison_norm = mpl.colors.BoundaryNorm(
        [-0.5, 0.5, 1.5, 2.5, 3.5], comparison_cmap.N
    )
    fig, ax = plt.subplots(figsize=(3.8, 5.8), dpi=EXPORT_DPI)
    ax.pcolormesh(
        LON_EDGES, LAT_EDGES,
        rasterize_column(comparison_values),
        cmap=comparison_cmap, norm=comparison_norm,
        shading='flat', rasterized=True
    )
    draw_geographic_context(ax)
    handles = [
        mpl.patches.Patch(
            facecolor='#4E6E8E', edgecolor='none',
            label=f'{TARGET_SHORT} only'
        ),
        mpl.patches.Patch(
            facecolor='#E1A45A', edgecolor='none',
            label=f'High P({TARGET_SHORT}) only'
        ),
        mpl.patches.Patch(
            facecolor='#9F2F3C', edgecolor='none',
            label=f'{TARGET_SHORT} and high P({TARGET_SHORT})'
        ),
    ]
    ax.legend(
        handles=handles, loc='center left',
        bbox_to_anchor=(1.02, 0.5), frameon=False
    )
    return save_figure(
        fig, 'best_extratrees_oof_true_slow_recovery_vs_high_probability_map'
    )

FEATURE_PRETTY_LABELS = {
    'logrho_0_15_mean': 'logrho 0-15 m',
    'logrho_15_30_mean': 'logrho 15-30 m',
    'logrho_30_50_mean': 'logrho 30-50 m',
    'logrho_50_100_mean': 'logrho 50-100 m',
    'logrho_100_150_mean': 'logrho 100-150 m',
    'logrho_150_200_mean': 'logrho 150-200 m',
    'logrho_200_300_mean': 'logrho 200-300 m',
    'logrho_300_400_mean': 'logrho 300-400 m',
    'logrho_shallow_0_50_mean': 'logrho shallow 0-50 m',
    'logrho_middle_50_150_mean': 'logrho middle 50-150 m',
    'logrho_deep_150_300_mean': 'logrho deep 150-300 m',
    'high_res_thickness_0_400_m': 'high-res. thickness 0-400 m',
    'continuous_high_res_thickness_m': 'continuous high-res. thickness',
    'low_res_thickness_0_400_m': 'low-res. thickness 0-400 m',
    'mid_low_barrier_strength': 'mid-depth low-res. barrier',
    'sand_clay_sand_contrast': 'sand-clay-sand contrast',
    'log1p_pumping_2012_drought_MayOct_mm': '2012 drought pumping',
    'log1p_pumping_mean_growing_2011_2023_mm_per_year': 'mean growing-season pumping',
    'log1p_pumping_mean_annual_2011_2023_mm_per_year': 'mean annual pumping',
    'log1p_pumping_peak_month_2011_2023_mm': 'peak monthly pumping',
}

def pretty_feature_name(feature: str) -> str:
    if feature in FEATURE_PRETTY_LABELS:
        return FEATURE_PRETTY_LABELS[feature]
    if feature.startswith('logrho_native_') and feature.endswith('m'):
        depth = feature.replace(
            'logrho_native_', ''
        ).replace('m', '').replace('p', '.')
        depth_value = float(depth)
        zmin = max(depth_value - 2.5, 0.0)
        zmax = depth_value + 2.5
        return f'rho {zmin:g}-{zmax:g} m'
    if feature == 'native_logrho_mean':
        return 'rho profile mean'
    if feature == 'native_logrho_std':
        return 'rho profile std'
    if feature == 'native_logrho_range':
        return 'rho profile range'
    return feature.replace('_', ' ')

def plot_feature_importance() -> Path:
    show_importance = importance.head(14).iloc[::-1]
    fig, ax = plt.subplots(figsize=(5.2, 4.2), dpi=EXPORT_DPI)
    ax.barh(
        np.arange(len(show_importance)),
        show_importance['importance'],
        color='#9F2F5E', edgecolor='none'
    )
    ax.set_yticks(np.arange(len(show_importance)))
    ax.set_yticklabels(
        [pretty_feature_name(v) for v in show_importance['feature']],
        fontsize=7.6
    )
    ax.set_xlabel('ExtraTrees feature importance')
    ax.grid(axis='x', color='#E2E2E2', lw=0.35)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.75)
    return save_figure(
        fig, 'best_extratrees_slow_recovery_feature_importance'
    )

fig_paths = [
    plot_probability_map(),
    plot_oof_overlap_map(),
    plot_feature_importance()
]
for path in fig_paths:
    print('Saved:', display_path(path))

